# Pandora Voice Design — ücretsiz VoxCPM2 çalışma defteri

Bu çalışma defteri yalnız **bir defalık özgün Pandora kadın sesi tasarımı** içindir.

- API anahtarı istemez.
- Google Drive erişimi istemez.
- Gerçek kişi veya ünlü sesi kullanmaz.
- Çıktıyı doğrudan `pandora_voice_candidates.zip` olarak indirir.
- Günlük Prometheus/Pandora konuşmaları bu ortamda üretilmez; yerel Chatterbox Multilingual V3 runtime kullanılır.

Çalışma sırası:

1. Hücreleri sırayla çalıştır.
2. 12 kısa adayı dinle.
3. `SHORTLIST` hücresine en fazla üç aday kimliği yaz.
4. Kalite paketini üret.
5. ZIP dosyasını indir.


In [ ]:
!nvidia-smi
!python -m pip install -q --upgrade "pip<26"

!python -m pip install -q \
  "voxcpm==2.0.3" \
  "huggingface-hub>=0.34,<1" \
  "soundfile>=0.12,<1" \
  "ipywidgets>=8,<9"

!python -m pip install -q --force-reinstall \
  "torch==2.11.0" \
  "torchvision==0.26.0" \
  "torchaudio==2.11.0" \
  --index-url https://download.pytorch.org/whl/cu128

!python -m pip install -q --force-reinstall "numpy==2.0.2"

print("Install complete. Restart the Colab runtime before continuing.")


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import sys
from datetime import datetime, timezone

import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from huggingface_hub import snapshot_download
from voxcpm import VoxCPM

assert torch.cuda.is_available(), "Colab GPU runtime seçilmedi."

MODEL_ID = "openbmb/VoxCPM2"
MODEL_REVISION = "bffb3df5a29440629464e5e839f4d214c8714c3d"
WORK = Path("/content/pandora_voice_design")
MODEL_DIR = WORK / "model"
STAGE_A = WORK / "stage_a"
EXPORT_ROOT = WORK / "export"
for path in (STAGE_A, EXPORT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

PERSONA = """An original Turkish female assistant voice, approximately late twenties.
Captivating, lively, warm and self-confident.
A subtly smoky timbre, but never breathy or whispery.
Clear contemporary Istanbul Turkish pronunciation.
Energetic conversational rhythm with natural pauses.
Intelligent, playful and approachable without sounding childish.
Medium pitch, expressive intonation and clean consonants.
Strong but friendly sentence endings.
Comfortable during long listening sessions.
Weather should sound practical and warm.
News should sound alert, neutral and credible.
Technical success should sound concise and confident.
Security warnings should become slower, firmer and serious.
Do not imitate any real person, actor, celebrity or existing assistant voice."""

REFERENCE_TEXT = (
    "Merhaba, ben Pandora. Canlı, sıcak ve güven veren sesimle bugün senin için buradayım. "
    "Sorularını dinlemeye ve birlikte çalışmaya hazırım."
)
TESTS = {
    "01_greeting": "Merhaba! Ben Pandora. Bugün senin için ne yapabilirim?",
    "02_weather": "İstanbul'da hava bugün değişken. Akşam saatlerinde yağmur ihtimali yükseliyor; dışarı çıkarken şemsiyeni almanı öneririm.",
    "03_news": "Bugünün teknoloji gündeminde üç önemli gelişme var. Önce en güncel ve doğrulanmış haberle başlayalım.",
    "04_technical_success": "Görev tamamlandı. Dört dosya değiştirildi ve tüm doğrulama testleri başarıyla geçti.",
    "05_security_warning": "Bu işlem bilgisayarındaki dosyaları değiştirecek. Devam etmek için ekrandaki onay düğmesine dokunmalısın.",
    "06_numbers_dates": "Bugün 3 Ağustos 2026. Saat 18.45. Toplam 478 testin 478'i başarıyla geçti.",
    "07_mixed_turkish_english": "FastAPI endpoint'i hazır. Git branch adı task-035a-pandora-local-voice ve doğrulama komutu python tire m pytest.",
    "08_long_form": "Prometheus Core, yazılım geliştirme görevlerini güvenli ve izlenebilir biçimde yönetmek için tasarlanmıştır. Pandora ise bu çekirdeğin mobil ve sesli arayüzüdür. Günlük sorulara doğal biçimde yanıt verir, güncel bilgileri kaynaklarıyla özetler ve teknik görevlerde yapılacak değişiklikleri önce açıkça anlatır. Dosya değiştiren veya komut çalıştıran işlemler yalnızca ekrandaki güvenli onaydan sonra ilerler. Böylece konuşmanın rahatlığı korunurken kritik eylemlerde denetim her zaman kullanıcıda kalır. Uzun bir işlem tamamlandığında Pandora önce kısa bir sesli özet verir, ayrıntıları ekranda gösterir ve başarısız olan doğrulamaları saklamadan bildirir."
}
MODE_CONTROL = {
    "01_greeting": "lively, warm, friendly, subtle smile",
    "02_weather": "warm, practical, energetic but calm",
    "03_news": "alert, neutral, credible",
    "04_technical_success": "concise, confident, positive",
    "05_security_warning": "slower, firm, serious, no smile",
    "06_numbers_dates": "clear, measured, precise articulation",
    "07_mixed_turkish_english": "clear Turkish with careful technical terms",
    "08_long_form": "comfortable long-form narration, natural pauses",
}

PERSONA_HASH = hashlib.sha256(PERSONA.encode("utf-8")).hexdigest()

snapshot = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    local_dir=MODEL_DIR,
)
model = VoxCPM.from_pretrained(str(snapshot), load_denoiser=False)
SAMPLE_RATE = int(model.tts_model.sample_rate)
print("VoxCPM2 loaded:", MODEL_REVISION, "sample_rate=", SAMPLE_RATE)


In [ ]:
# Stage A: generate 12 deterministic candidate references.

import random

SEEDS = list(range(42, 54))
stage_a_rows = []

for index, seed in enumerate(SEEDS, start=1):
    candidate_id = f"pandora-{index:02d}"
    path = STAGE_A / f"{candidate_id}.wav"

    if not path.exists():
        designed_text = f"({PERSONA}){REFERENCE_TEXT}"

        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        wav = model.generate(
            text=designed_text,
            cfg_value=2.0,
            inference_timesteps=10,
            normalize=True,
            denoise=False,
            retry_badcase=True,
            retry_badcase_max_times=2,
        )

        sf.write(
            path,
            np.asarray(wav, dtype=np.float32),
            SAMPLE_RATE,
        )

    stage_a_rows.append((candidate_id, seed, path))

for candidate_id, seed, path in stage_a_rows:
    print(candidate_id, "seed=", seed)
    display(Audio(filename=str(path)))


## Shortlist seçimi

Aşağıdaki hücrede yalnız dinlediğin adaylardan **1–3** tanesini bırak.

Örnek:

```python
SHORTLIST = ["pandora-03", "pandora-08"]
```


In [ ]:
SHORTLIST = ["pandora-01"]  # Dinledikten sonra bunu değiştir.

valid_ids = {candidate_id for candidate_id, _, _ in stage_a_rows}
assert 1 <= len(SHORTLIST) <= 3, "SHORTLIST 1–3 aday içermeli."
assert len(set(SHORTLIST)) == len(SHORTLIST), "SHORTLIST tekrar içeremez."
assert set(SHORTLIST) <= valid_ids, "Bilinmeyen aday kimliği var."
print("Shortlist:", SHORTLIST)


In [ ]:
import random

# Aşama B: seçilen her aday için tam kalite paketi.
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
EXPORT_ROOT.mkdir(parents=True)

manifest_candidates = []
seed_by_id = {candidate_id: seed for candidate_id, seed, _ in stage_a_rows}
preview_by_id = {candidate_id: path for candidate_id, _, path in stage_a_rows}

for candidate_id in SHORTLIST:
    seed = seed_by_id[candidate_id]
    candidate_dir = EXPORT_ROOT / "candidates" / candidate_id
    candidate_dir.mkdir(parents=True)

    reference_path = candidate_dir / "reference.wav"
    shutil.copyfile(preview_by_id[candidate_id], reference_path)

    clips = {}
    for offset, (category, text) in enumerate(TESTS.items(), start=1):
        clip_path = candidate_dir / f"{category}.wav"
        controlled_text = f"({MODE_CONTROL[category]}){text}"

        clip_seed = seed * 100 + offset

        random.seed(clip_seed)
        np.random.seed(clip_seed)
        torch.manual_seed(clip_seed)
        torch.cuda.manual_seed_all(clip_seed)

        wav = model.generate(
            text=controlled_text,
            reference_wav_path=str(reference_path),
            cfg_value=2.0,
            inference_timesteps=10,
            normalize=True,
            denoise=False,
            retry_badcase=True,
            retry_badcase_max_times=2,
        )
        sf.write(clip_path, np.asarray(wav, dtype=np.float32), SAMPLE_RATE)
        clips[category] = {
            "path": clip_path.relative_to(EXPORT_ROOT).as_posix(),
            "sha256": sha256_file(clip_path),
        }

    manifest_candidates.append({
        "candidate_id": candidate_id,
        "seed": seed,
        "persona_hash": PERSONA_HASH,
        "model_revision": MODEL_REVISION,
        "reference": {
            "path": reference_path.relative_to(EXPORT_ROOT).as_posix(),
            "sha256": sha256_file(reference_path),
            "transcript": REFERENCE_TEXT,
        },
        "clips": clips,
    })

manifest = {
    "schema_version": 2,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "persona_hash": PERSONA_HASH,
    "sample_rate": SAMPLE_RATE,
    "candidates": manifest_candidates,
}
(EXPORT_ROOT / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
(EXPORT_ROOT / "persona.txt").write_text(PERSONA + "\n", encoding="utf-8")
(EXPORT_ROOT / "model_metadata.json").write_text(
    json.dumps({
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "license": "Apache-2.0",
        "voxcpm_package": __import__("importlib.metadata").metadata.version("voxcpm"),
        "python": sys.version,
        "torch": torch.__version__,
    }, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

for candidate in manifest_candidates:
    print("\\n", candidate["candidate_id"])
    base = EXPORT_ROOT / "candidates" / candidate["candidate_id"]
    for category in TESTS:
        print(category)
        display(Audio(filename=str(base / f"{category}.wav")))


In [ ]:
# ZIP oluştur ve indir.
archive_base = Path("/content/pandora_voice_candidates")
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=EXPORT_ROOT))
print("Created:", archive_path, archive_path.stat().st_size, "bytes")

from google.colab import files
files.download(str(archive_path))
